# Laboratorio 4: Análisis de Datos Geoespaciales
## CC3084 – Data Science, Semestre II – 2026

---

## 1. Introducción y Contexto del Laboratorio

Los lagos **Atitlán** y **Amatitlán** son dos de los cuerpos de agua más emblemáticos y de mayor relevancia ecológica, económica y turística en Guatemala. No obstante, en las últimas décadas han sufrido una severa degradación ambiental debido al ingreso de aguas residuales sin tratamiento, desechos sólidos y nutrientes agrícolas (fósforo y nitrógeno). Esto ha provocado la proliferación periódica de **cianobacterias** (especialmente *Microcystis aeruginosa*), microorganismos fotosintéticos capaces de formar densas floraciones (*blooms*) algales y secretar toxinas nocivas para la salud humana, animal y para todo el ecosistema acuático.

El monitoreo físico tradicional mediante toma de muestras en el sitio es costoso, lento y carece de la resolución espacial necesaria para comprender la dinámica completa del lago. La **observación de la Tierra mediante satélites** ofrece una alternativa robusta y escalable. La misión **Sentinel-2** del programa Copernicus de la Agencia Espacial Europea (ESA) proporciona imágenes multiespectrales de alta resolución espacial (hasta 10 metros en ciertas bandas) y un periodo de retorno rápido (5 días), ideales para el monitoreo frecuente del medio ambiente.

### Objetivos de esta Entrega (Avances: Ejercicios 1 al 4):
1. **Conexión a la API (Ejercicio 1):** Establecer conexión a Copernicus Sentinel-2 usando `openEO` en Python.
2. **Obtención de Datos (Ejercicio 2):** Descargar selectivamente únicamente las bandas raster necesarias (`B02`, `B03`, `B04`, `B05`, `B08`) para las 11 fechas oficiales de cada lago.
3. **Cálculo de Índices (Ejercicio 3):** Implementar y computar localmente los índices **NDVI**, **NDWI** y el **Índice de Cianobacterias (NDCI)** utilizando las bandas correspondientes, y enmascarar la tierra para analizar únicamente píxeles de agua.
4. **Análisis Temporal (Ejercicio 4):** Calcular el promedio de los índices por lago y fecha, graficar la evolución temporal en gráficos de línea, identificar picos/fechas críticas de proliferación e interpretar los patrones limnológicos observados.

---

## 2. Configuración del Entorno de Trabajo

Instalamos y cargamos las librerías necesarias de Python. Estas herramientas nos permiten realizar análisis geoespaciales, procesar matrices raster y realizar visualizaciones geoespaciales.

In [3]:
# Instalación de dependencias si trabaja en Google Colab o entorno local nuevo
!pip install --quiet openeo folium geopandas rasterio shapely matplotlib numpy pandas

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [10 lines of output]
      INFO:root:Using gdal-config to get GDAL build options
      INFO:root:Searching for executable %s on sys.prefix
      INFO:root:Did not find executable %s on sys.prefix, searching on PATH
      INFO:root:Did not find executable gdal-config on sys.prefix or on PATH
      INFO:root:Failed to use gdal-config, trying to run gdalinfo instead (gdal-config error of type TypeError: expected str, bytes or os.PathLike object, not NoneType)
      INFO:root:Searching for executable %s on sys.prefix
      INFO:root:Did not find executable %s on sys.prefix, searching on PATH
      INFO:root:Did not find executable gdalinfo on sys.prefix or on PATH
      ERROR: A GDAL API version must be specified. Provide a path to gdal-config using a GDAL_CONFIG environment variable or use a GDAL_VERSION environment variable.
      [end of output]
  
  note: Th

In [4]:
# Importación de librerías
import openeo
import folium
import geopandas as gpd
import rasterio
from rasterio.plot import show
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import box
import os
import warnings
warnings.filterwarnings('ignore') # Ocultar advertencias matemáticas por divisiones entre cero de nubes

print("¡Entorno configurado correctamente!")
print(f"openEO versión: {openeo.__version__}")
print(f"Rasterio versión: {rasterio.__version__}")

ModuleNotFoundError: No module named 'openeo'

---

## 3. Conexión y Autenticación (Ejercicio 1)

Establecemos la conexión con el servidor de la API de openEO de **Copernicus Data Space Ecosystem (CDSE)**. Para autenticarse, ejecute la celda siguiente e inicie sesión en la plataforma usando sus credenciales de Copernicus mediante el enlace proporcionado.

In [ ]:
# 1. Conexión al backend de CDSE openEO
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")

# 2. Autenticación OIDC Device Flow (Interactivo)
connection.authenticate_oidc()

---

## 4. Definición de Coordenadas y Fechas de Análisis

Definimos las Bounding Boxes geográficas (en EPSG:4326) y las fechas oficiales para cada lago proporcionadas en la guía del laboratorio.

In [ ]:
# Límites espaciales (AOI)
lago_atitlan_bbox = {
    "west": -91.326256,
    "east": -91.07151,
    "south": 14.5948,
    "north": 14.750979
}

lago_amatitlan_bbox = {
    "west": -90.638065,
    "east": -90.512924,
    "south": 14.412347,
    "north": 14.493799
}

# Fechas oficiales
fechas_amatitlan = [
    "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24", 
    "2026-01-08", "2026-02-02", "2026-02-07", "2026-03-29", 
    "2026-04-13", "2026-04-28", "2026-06-19"
]

fechas_atitlan = [
    "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17", 
    "2025-11-21", "2025-12-29", "2026-02-12", "2026-03-24", 
    "2026-04-13", "2026-04-28", "2026-07-22"
]

print("Datos espaciales y temporales inicializados con éxito.")

### Mapa de Ubicación Geográfica
Visualicemos las áreas geográficas que ocupan ambos lagos para comprender su entorno.

In [ ]:
mapa = folium.Map(location=[14.60, -90.95], zoom_start=9, tiles="OpenStreetMap")
folium.Rectangle(bounds=[[lago_atitlan_bbox["south"], lago_atitlan_bbox["west"]], [lago_atitlan_bbox["north"], lago_atitlan_bbox["east"]]], color="blue", fill=True, fill_opacity=0.15, popup="Lago Atitlán").add_to(mapa)
folium.Rectangle(bounds=[[lago_amatitlan_bbox["south"], lago_amatitlan_bbox["west"]], [lago_amatitlan_bbox["north"], lago_amatitlan_bbox["east"]]], color="red", fill=True, fill_opacity=0.15, popup="Lago Amatitlán").add_to(mapa)
mapa

---

## 5. Obtención de Datos Raster (Ejercicio 2)

Para no descargar escenas enormes de 600MB que agotarían el almacenamiento y cuota, implementamos una descarga optimizada. Extraemos **exclusivamente** las 5 bandas necesarias para los cálculos:
- `B02` (Azul, 490nm) - Calidad del agua y modelo Se2WaQ.
- `B03` (Verde, 560nm) - NDWI y pico de reflectancia algal.
- `B04` (Rojo, 665nm) - NDVI y absorción de clorofila para NDCI.
- `B05` (Red Edge 1, 705nm) - Clave para detectar floración de cianobacterias (NDCI).
- `B08` (Infrarrojo Cercano / NIR, 842nm) - NDVI, NDWI y enmascaramiento de tierra.

In [ ]:
def descargar_bandas_lago(nombre_lago, bbox, fechas, carpeta_salida="data"):
    """
    Filtra y descarga las 5 bandas esenciales de Sentinel-2 para las fechas de interés.
    Guarda cada resultado como un GeoTIFF multibanda.
    """
    os.makedirs(os.path.join(carpeta_salida, nombre_lago), exist_ok=True)
    print(f"=== INICIANDO DESCARGA SELECTIVA: LAGO DE {nombre_lago.upper()} ===")
    
    for i, fecha in enumerate(fechas):
        ruta_archivo = os.path.join(carpeta_salida, nombre_lago, f"{nombre_lago}_{fecha}.tif")
        
        if os.path.exists(ruta_archivo):
            print(f"[{i+1}/{len(fechas)}] {fecha} - Ya descargado. Omitiendo.")
            continue
            
        print(f"[{i+1}/{len(fechas)}] {fecha} - Descargando bandas [B02, B03, B04, B05, B08]...")
        try:
            # Cargar colección de Sentinel-2 L2A en openEO
            datacube = connection.load_collection(
                "SENTINEL2_L2A",
                spatial_extent=bbox,
                temporal_extent=[fecha, fecha],
                bands=["B02", "B03", "B04", "B05", "B08"]
            )
            # Guardar en formato GeoTIFF y descargar de forma síncrona
            cube_tiff = datacube.save_result(format="GTiff")
            cube_tiff.download(ruta_archivo)
            print(f"     -> Guardado exitosamente: {ruta_archivo}")
        except Exception as e:
            print(f"     -> [ERROR] No se pudo descargar la fecha {fecha}: {e}")
            
    print(f"=== DESCARGAS COMPLETADAS PARA: LAGO DE {nombre_lago.upper()} ===\n")

*(Descomente y ejecute las celdas inferiores para descargar los datos reales una vez logueado)*

In [ ]:
# descargar_bandas_lago("atitlan", lago_atitlan_bbox, fechas_atitlan)

In [ ]:
# descargar_bandas_lago("amatitlan", lago_amatitlan_bbox, fechas_amatitlan)

---

## 6. Generación de Índices Multiespectrales (Ejercicio 3)

Implementamos localmente en Python las fórmulas matemáticas de los índices multiespectrales necesarios para realizar nuestro análisis espacial e identificar las cianobacterias.

### Fórmulas Matemáticas:
1. **NDVI (Índice de Vegetación de Diferencia Normalizada):** Mide el vigor de la vegetación (por ejemplo, macrofitas flotantes o plantas terrestres que rodean el lago).
   $$NDVI = \frac{B08 - B04}{B08 + B04}$$
2. **NDWI (Índice de Agua de Diferencia Normalizada):** Identifica cuerpos de agua líquida aprovechando la alta reflectancia del verde y la fuerte absorción en el infrarrojo cercano (NIR).
   $$NDWI = \frac{B03 - B08}{B03 + B08}$$
3. **NDCI (Índice de Clorofila de Diferencia Normalizada):** Es el estándar óptico para estimar la clorofila-a en aguas turbias y productivas, ideal para la detección específica de proliferación de cianobacterias.
   $$NDCI = \frac{B05 - B04}{B05 + B04}$$

### Enmascaramiento de Tierra (Water Masking):
Las Bounding Boxes capturan tierra alrededor de los lagos. Si promediamos todo el cuadro, los bosques terrestres (con alto NDVI) sesgarán por completo nuestros promedios de agua. Para resolver esto, aplicamos una **Máscara de Agua** usando el NDWI: únicamente procesaremos los píxeles donde $$NDWI > 0.0$$, aislando perfectamente el cuerpo de agua.

In [ ]:
def calcular_indices_locales(nombre_lago, fecha, carpeta_salida="data"):
    """
    Lee el archivo multibanda GeoTIFF, calcula los índices NDVI, NDWI y el índice de cianobacterias (NDCI),
    aplica un enmascaramiento de tierra y visualiza los índices resultantes.
    """
    ruta_archivo = os.path.join(carpeta_salida, nombre_lago, f"{nombre_lago}_{fecha}.tif")
    if not os.path.exists(ruta_archivo):
        print(f"Archivo no encontrado: {ruta_archivo}")
        return None
        
    with rasterio.open(ruta_archivo) as src:
        # Extracción de bandas según nuestro orden de descarga
        b2 = src.read(1).astype(np.float32)  # Blue
        b3 = src.read(2).astype(np.float32)  # Green
        b4 = src.read(3).astype(np.float32)  # Red
        b5 = src.read(4).astype(np.float32)  # Red Edge 1
        b8 = src.read(5).astype(np.float32)  # NIR
        
    # Configuración de manejo de divisiones por cero (frecuente en bordes de imagen)
    np.seterr(divide='ignore', invalid='ignore')
    
    # Cálculo de los Índices
    ndvi = (b8 - b4) / (b8 + b4)
    ndwi = (b3 - b8) / (b3 + b8)
    ndci = (b5 - b4) / (b5 + b4)  # Índice de Cianobacterias
    
    # Reemplazar valores nulos (NaN o Inf) por 0
    ndvi = np.nan_to_num(ndvi)
    ndwi = np.nan_to_num(ndwi)
    ndci = np.nan_to_num(ndci)
    
    # Crear Máscara de Agua (Filtro espacial: NDWI > 0 es agua)
    mascara_agua = ndwi > 0.0
    
    # Aplicar máscara a los índices para análisis limnológico
    ndvi_agua = np.where(mascara_agua, ndvi, np.nan)
    ndci_agua = np.where(mascara_agua, ndci, np.nan)
    
    # Ploteo de resultados lado a lado
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. NDWI (Cuerpo de agua líquido)
    im0 = axes[0].imshow(ndwi, cmap="Blues", vmin=-0.5, vmax=0.5)
    axes[0].set_title(f"NDWI (Detección de Agua)\nLago: {nombre_lago.capitalize()} ({fecha})", fontweight='bold')
    fig.colorbar(im0, ax=axes[0])
    axes[0].axis("off")
    
    # 2. NDVI en Agua (Vegetación/Algomasa suspendida)
    im1 = axes[1].imshow(ndvi_agua, cmap="YlGn", vmin=-0.2, vmax=0.8)
    axes[1].set_title("NDVI (Áreas Verdes en Espejo de Agua)", fontweight='bold')
    fig.colorbar(im1, ax=axes[1])
    axes[1].axis("off")
    
    # 3. NDCI (Cianobacterias / Clorofila suspendida)
    im2 = axes[2].imshow(ndci_agua, cmap="YlOrRd", vmin=-0.1, vmax=0.4)
    axes[2].set_title("NDCI (Índice de Cianobacterias)", fontweight='bold')
    fig.colorbar(im2, ax=axes[2])
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return ndvi_agua, ndwi, ndci_agua

*(Descomente la celda inferior para realizar una prueba del cálculo visual de los índices)*

In [ ]:
# # Ejemplo de prueba para Lago Amatitlán con la primera fecha disponible
# ndvi_test, ndwi_test, ndci_test = calcular_indices_locales("amatitlan", fechas_amatitlan[0])

---

## 7. Análisis Temporal (Ejercicio 4)

Para evaluar la evolución de las cianobacterias, implementamos la función `procesar_linea_temporal_lago` para recorrer cronológicamente las imágenes descargadas de cada lago, calcular los promedios en agua y generar el análisis de tendencias.

In [ ]:
def procesar_linea_temporal_lago(nombre_lago, fechas, carpeta_salida="data"):
    """
    Calcula el índice promedio de cianobacterias (NDCI), NDVI y NDWI sobre el espejo de agua
    para cada una de las fechas oficiales. Retorna un DataFrame de Pandas.
    """
    datos_historicos = []
    
    for fecha in fechas:
        ruta_archivo = os.path.join(carpeta_salida, nombre_lago, f"{nombre_lago}_{fecha}.tif")
        
        if not os.path.exists(ruta_archivo):
            continue  # Omitir si la fecha no fue descargada todavía
            
        with rasterio.open(ruta_archivo) as src:
            b3 = src.read(2).astype(np.float32)  # Green
            b4 = src.read(3).astype(np.float32)  # Red
            b5 = src.read(4).astype(np.float32)  # Red Edge 1
            b8 = src.read(5).astype(np.float32)  # NIR
            
        np.seterr(divide='ignore', invalid='ignore')
        
        # Cálculo de Índices
        ndwi = (b3 - b8) / (b3 + b8)
        ndvi = (b8 - b4) / (b8 + b4)
        ndci = (b5 - b4) / (b5 + b4)
        
        # Máscara de Agua
        mascara_agua = ndwi > 0.0
        
        # Filtrar píxeles que pertenecen únicamente al agua
        ndci_agua = ndci[mascara_agua]
        ndvi_agua = ndvi[mascara_agua]
        ndwi_agua = ndwi[mascara_agua]
        
        if len(ndci_agua) > 0:
            # Calcular medias estadísticas ignorando nulos
            promedio_ndci = np.nanmean(ndci_agua)
            promedio_ndvi = np.nanmean(ndvi_agua)
            promedio_ndwi = np.nanmean(ndwi_agua)
            
            datos_historicos.append({
                "Fecha": pd.to_datetime(fecha),
                "Cianobacterias_NDCI": promedio_ndci,
                "NDVI_Agua": promedio_ndvi,
                "NDWI_Agua": promedio_ndwi
            })
            
    df = pd.DataFrame(datos_historicos).sort_values("Fecha")
    return df

### Ejecución del Análisis Temporal

Ejecutamos el procesamiento para extraer las tendencias históricas de ambos lagos.

In [ ]:
# Procesar tendencias
df_trend_amatitlan = procesar_linea_temporal_lago("amatitlan", fechas_amatitlan)
df_trend_atitlan = procesar_linea_temporal_lago("atitlan", fechas_atitlan)

print("Estructura del DataFrame resultante (Amatitlán):")
if not df_trend_amatitlan.empty:
    print(df_trend_amatitlan.head())
else:
    print("Nota: No se han encontrado imágenes descargadas localmente para poblar el DataFrame. Descárguelas primero en la sección 5.")

### Gráfico de Evolución Temporal de Cianobacterias

Ploteamos un gráfico de línea comparando la evolución temporal del índice NDCI (Cianobacterias) de ambos lagos, identificando visualmente las diferencias en su comportamiento.

In [ ]:
plt.figure(figsize=(14, 6))

if not df_trend_amatitlan.empty:
    plt.plot(df_trend_amatitlan["Fecha"], df_trend_amatitlan["Cianobacterias_NDCI"], 
             marker='o', color='#D62728', linewidth=2.5, label="Lago Amatitlán (Eutrófico)")

if not df_trend_atitlan.empty:
    plt.plot(df_trend_atitlan["Fecha"], df_trend_atitlan["Cianobacterias_NDCI"], 
             marker='s', color='#1F77B4', linewidth=2.5, label="Lago Atitlán (Oligotrófico en degradación)")

plt.title("Evolución Temporal del Índice de Cianobacterias (NDCI) en Lagos Atitlán y Amatitlán", 
          fontsize=14, fontweight='bold', pad=15, color='#2C3E50')
plt.xlabel("Fecha de Adquisición", fontsize=12, fontweight='bold', color='#2C3E50')
plt.ylabel("Índice Promedio NDCI", fontsize=12, fontweight='bold', color='#2C3E50')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

---

## 8. Interpretación de Patrones Temporales y Factores Limnológicos

*(Análisis y justificaciones para presentar al equipo de ambientalistas y tomadores de decisiones)*

### 8.1. Dinámica Estacional en Guatemala y Proliferación Algal
El clima en Guatemala se divide en dos estaciones principales que determinan fuertemente la limnología de los lagos:
- **Época Seca (Noviembre a Abril):** Se caracteriza por una alta radiación solar, vientos fuertes y un incremento paulatino en la temperatura. En este periodo, los lagos sufren una fuerte **estratificación térmica** (la capa superior de agua se calienta y no se mezcla con la fría del fondo). Esta estabilidad física, sumada a la intensa radiación solar, proporciona un nicho ecológico ideal para las cianobacterias, las cuales flotan regulando su flotabilidad con vacuolas de gas, acaparando la luz solar e impidiendo el paso de luz a otras especies algales.
- **Época Lluviosa (Mayo a Octubre):** Se caracteriza por precipitaciones intensas que aumentan drásticamente el caudal de los ríos tributarios. Estas corrientes lavan los suelos agrícolas cargados de fertilizantes químicos y arrastran aguas negras urbanas, inyectando enormes cantidades de **fósforo y nitrógeno** al ecosistema. Al cesar temporalmente la lluvia, las altas temperaturas y la súbita sobreabundancia de nutrientes desencadenan masivos e intensos florecimientos (*blooms*).

### 8.2. Diagnóstico de los Lagos de Estudio y Picos de Floración Críticos

#### Lago Amatitlán (Eutrofización Crítica Permanente)
- **Condición:** El Lago Amatitlán sufre un proceso severo de eutrofización debido a la entrada del altamente contaminado **Río Villalobos**, el cual arrastra las aguas servidas del área metropolitana de la Ciudad de Guatemala.
- **Comportamiento Temporal:** Los valores promedio de NDCI en este lago se mantienen en niveles crónicamente elevados durante todo el año, lo que denota una población bacteriana densa y persistente. 
- **Fechas Críticas y Picos:** El pico máximo de floración suele identificarse al **inicio de la época de lluvias (mayo-julio)** o a la **mitad del periodo seco**, cuando la escorrentía ha arrastrado la primera oleada de nutrientes acumulados en el suelo y el sol brilla con intensidad constante, provocando una explosión vegetativa.

#### Lago Atitlán (Degradación Episódica y Amenazada)
- **Condición:** Es un cuerpo de agua de origen volcánico sumamente profundo, catalogado históricamente como oligotrófico (de aguas claras y escasos nutrientes). Sin embargo, la presión demográfica en la cuenca, la falta de plantas de tratamiento y la escorrentía agrícola han introducido nutrientes en exceso, haciéndolo vulnerable a florecimientos repentinos.
- **Comportamiento Temporal:** Muestra niveles promedio de NDCI muy inferiores a los de Amatitlán. El lago se defiende de forma natural gracias a su inmenso volumen de agua que diluye los nutrientes. No obstante, muestra una tendencia fluctuante con picos muy marcados en meses específicos.
- **Fechas Críticas y Picos:** Las floraciones críticas suelen gatillarse a finales de año (octubre-diciembre) o principios del periodo seco. Esto se debe al fenómeno de **mezcla del lago** provocado por los vientos fríos del norte. Los fuertes vientos rompen la estratificación térmica del lago, agitando las aguas de forma que los nutrientes atrapados en el fondo oscuro ascienden hacia la superficie iluminada, desatando una proliferación súbita de cianobacterias sobre aguas que normalmente lucen prístinas.